# Sales Forecasting — Baseline
**Goal:** Predict daily `Revenue` and `COGS` for 2023-01-01 → 2024-07-01 using historical data (2012–2022).

**Strategy (simple seasonal average + trend):**
1. Compute average YoY growth rate from 2013–2022.
2. Build a "seasonal profile" — the average Revenue/COGS for each calendar day-of-year across all historical years.
3. Scale the profile by the projected year-level trend to produce predictions.

## 1 — Imports & Config

In [25]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from xgboost import XGBRegressor

# ===== LOAD =====
DATA_DIR = 'dataset/'

products = pd.read_csv(DATA_DIR+'master/products.csv')
orders = pd.read_csv(DATA_DIR+'transaction/orders.csv')
order_items = pd.read_csv(DATA_DIR+'transaction/order_items.csv')
promotions = pd.read_csv(DATA_DIR+'master/promotions.csv')
inventory = pd.read_csv(DATA_DIR+'operational/inventory.csv')
web = pd.read_csv(DATA_DIR+'operational/web_traffic.csv')
sales = pd.read_csv(DATA_DIR+'analytical/sales.csv')

# ===== DATE =====
orders['order_date'] = pd.to_datetime(orders['order_date'])
web['date'] = pd.to_datetime(web['date'])
sales['Date'] = pd.to_datetime(sales['Date'])
inventory['snapshot_date'] = pd.to_datetime(inventory['snapshot_date'])
promotions['start_date'] = pd.to_datetime(promotions['start_date'])
promotions['end_date'] = pd.to_datetime(promotions['end_date'])

2. FEATURE ENGINEERING

In [26]:
def build_features():
    
    # ===== FACT =====
    df = order_items.merge(products, on='product_id', how='left')
    df = df.merge(orders[['order_id','order_date']], on='order_id', how='left')
    df.rename(columns={'order_date':'Date'}, inplace=True)

    df['gross'] = df['quantity'] * df['unit_price']
    df['net'] = df['gross'] - df['discount_amount']
    df['cogs_total'] = df['quantity'] * df['cogs']

    # ===== DAILY =====
    daily = df.groupby('Date').agg({
        'order_id':'nunique',
        'quantity':'sum',
        'net':'sum',
        'cogs_total':'sum',
        'discount_amount':'sum',
        'unit_price':'mean'
    }).reset_index()

    daily.columns = ['Date','num_orders','qty','Revenue','COGS','discount','avg_price']

    # ===== WEB =====
    web_df = web.rename(columns={'date':'Date'})
    daily = daily.merge(web_df, on='Date', how='left')

    daily['conversion'] = daily['num_orders'] / (daily['sessions'] + 1e-6)

    # ===== INVENTORY =====
    inv = inventory.groupby('snapshot_date').agg({
        'stock_on_hand':'sum',
        'stockout_flag':'mean',
        'fill_rate':'mean'
    }).reset_index()

    inv.rename(columns={'snapshot_date':'Date'}, inplace=True)
    inv = inv.set_index('Date').resample('D').ffill().reset_index()

    daily = daily.merge(inv, on='Date', how='left')

    # ===== PROMO =====
    promo_list = []
    for _, row in promotions.iterrows():
        dates = pd.date_range(row['start_date'], row['end_date'])
        tmp = pd.DataFrame({'Date':dates})
        tmp['promo'] = 1
        tmp['discount_val'] = row['discount_value']
        promo_list.append(tmp)

    promo_df = pd.concat(promo_list)
    promo_daily = promo_df.groupby('Date').agg({
        'promo':'sum',
        'discount_val':'mean'
    }).reset_index()

    daily = daily.merge(promo_daily, on='Date', how='left')

    daily.fillna(0, inplace=True)

    # ===== TIME =====
    daily['year'] = daily['Date'].dt.year
    daily['month'] = daily['Date'].dt.month
    daily['day'] = daily['Date'].dt.day
    daily['dow'] = daily['Date'].dt.dayofweek
    daily['week'] = daily['Date'].dt.isocalendar().week.astype(int)
    daily['quarter'] = daily['Date'].dt.quarter
    daily['is_weekend'] = (daily['dow']>=5).astype(int)

    # ===== INTERACTION =====
    daily['traffic_x_conversion'] = daily['sessions'] * daily['conversion']
    daily['promo_x_traffic'] = daily['promo'] * daily['sessions']
    daily['stockout_x_demand'] = daily['stockout_flag'] * daily['num_orders']

    # ===== LAG =====
    daily = daily.sort_values('Date')

    for lag in [1,7,14,30]:
        daily[f'lag_{lag}'] = daily['Revenue'].shift(lag)

    daily['roll7'] = daily['Revenue'].rolling(7).mean().shift(1)
    daily['roll30'] = daily['Revenue'].rolling(30).mean().shift(1)

    return daily

3. BUILD DATASET

In [27]:
df = build_features()

print("DATE RANGE:", df['Date'].min(), df['Date'].max())

train = df[df['Date'] <= '2022-12-31'].copy()
test  = df[df['Date'] >= '2023-01-01'].copy()

if test.empty:
    raise ValueError("❌ TEST EMPTY — dataset không có data 2023+")

DATE RANGE: 2012-07-04 00:00:00 2022-12-31 00:00:00


ValueError: ❌ TEST EMPTY — dataset không có data 2023+

BASELINE (SEASONAL + TREND)

In [ ]:
annual = train.groupby('year')[['Revenue','COGS']].sum()

full_years = annual.loc[2013:2022]

growth_rev = (1 + full_years['Revenue'].pct_change().dropna()).prod() ** (1/9)
growth_cogs = (1 + full_years['COGS'].pct_change().dropna()).prod() ** (1/9)

base_rev  = annual.loc[2022, 'Revenue'] / 365
base_cogs = annual.loc[2022, 'COGS'] / 365

# ===== NORMALIZE =====
annual_means = train.groupby('year')[['Revenue','COGS']].transform('mean')
train['rev_norm'] = train['Revenue'] / annual_means['Revenue']
train['cogs_norm'] = train['COGS'] / annual_means['COGS']

seasonal = train.groupby(['month','day'])[['rev_norm','cogs_norm']].mean().reset_index()

# ===== APPLY =====
test = test.merge(seasonal, on=['month','day'], how='left')
test[['rev_norm','cogs_norm']] = test[['rev_norm','cogs_norm']].fillna(1)

test['years_ahead'] = test['year'] - 2022

test['Revenue_pred'] = base_rev * growth_rev**test['years_ahead'] * test['rev_norm']
test['COGS_pred'] = base_cogs * growth_cogs**test['years_ahead'] * test['cogs_norm']

ML (RESIDUAL MODEL)

In [ ]:
features = [
    'dow','week','month','quarter','is_weekend',
    'lag_1','lag_7','lag_14','lag_30','roll7','roll30',
    'sessions','conversion',
    'stockout_flag','fill_rate',
    'promo','discount_val',
    'traffic_x_conversion',
    'promo_x_traffic',
    'stockout_x_demand'
]

train_ml = train.merge(seasonal, on=['month','day'], how='left')
train_ml['rev_norm'] = train_ml['rev_norm'].fillna(1)

train_ml['baseline'] = base_rev * growth_rev**(train_ml['year']-2022) * train_ml['rev_norm']
train_ml['residual'] = train_ml['Revenue'] - train_ml['baseline']

train_ml = train_ml.dropna()

model = XGBRegressor(
    n_estimators=800,
    learning_rate=0.02,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8
)

model.fit(train_ml[features], train_ml['residual'])

LAG FOR TEST

In [ ]:
last = train.sort_values('Date').tail(30)

for lag in [1,7,14,30]:
    test[f'lag_{lag}'] = last['Revenue'].iloc[-lag]

test['roll7'] = last['Revenue'].tail(7).mean()
test['roll30'] = last['Revenue'].tail(30).mean()

FINAL PREDICTION

In [ ]:
residual_pred = model.predict(test[features])

test['Revenue_final'] = test['Revenue_pred'] + residual_pred

submission = pd.DataFrame({
    'Date': test['Date'].dt.strftime('%Y-%m-%d'),
    'Revenue': test['Revenue_final'].round(2),
    'COGS': test['COGS_pred'].round(2)
})

submission.to_csv('submission.csv', index=False)

print("DONE ✅")
print(submission.head())

In [18]:
# ===== STEP 1: CREATE NORMALIZED FEATURE =====
annual_means = train.groupby('year')[['Revenue','COGS']].transform('mean')

train['rev_norm']  = train['Revenue'] / annual_means['Revenue']
train['cogs_norm'] = train['COGS'] / annual_means['COGS']


# ===== STEP 2: BUILD SEASONAL (ĐÚNG) =====
seasonal = (
    train
    .groupby(['month','day'])[['rev_norm','cogs_norm']]
    .mean()
    .reset_index()
)

print("CHECK seasonal columns:", seasonal.columns)


# ===== STEP 3: MERGE TO TEST (SAFE) =====
test = test.merge(
    seasonal,
    on=['month','day'],
    how='left'
)


# ===== STEP 4: SAFE FILL (KHÔNG BAO GIỜ CRASH) =====
for col in ['rev_norm', 'cogs_norm']:
    if col not in test.columns:
        test[col] = 1
    else:
        test[col] = test[col].fillna(1)


# ===== STEP 5: BASELINE PRED =====
test['years_ahead'] = test['year'] - 2022

test['Revenue_pred'] = base_rev * growth_rev**test['years_ahead'] * test['rev_norm']
test['COGS_pred']    = base_cogs * growth_cogs**test['years_ahead'] * test['cogs_norm']

CHECK seasonal columns: Index(['month', 'day', 'rev_norm', 'cogs_norm'], dtype='object')


In [20]:
# ===== TRAIN ML =====
train_ml = train.copy()

# merge seasonal (TRÁNH overwrite)
train_ml = train_ml.merge(
    seasonal,
    on=['month','day'],
    how='left',
    suffixes=('', '_seasonal')
)

# ===== FIX COLUMN rev_norm =====
if 'rev_norm' not in train_ml.columns:
    if 'rev_norm_seasonal' in train_ml.columns:
        train_ml['rev_norm'] = train_ml['rev_norm_seasonal']
    else:
        train_ml['rev_norm'] = 1

train_ml['rev_norm'] = train_ml['rev_norm'].fillna(1)

# ===== BASELINE =====
train_ml['baseline_rev'] = (
    base_rev * growth_rev**(train_ml['year'] - 2022) * train_ml['rev_norm']
)

train_ml['residual_rev'] = train_ml['Revenue'] - train_ml['baseline_rev']

# 👉 dropna SAU KHI build xong feature
train_ml = train_ml.dropna().copy()

In [21]:
from xgboost import XGBRegressor

model = XGBRegressor(
    n_estimators=500,
    learning_rate=0.03,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8
)

model.fit(train_ml[features_ml], train_ml['residual_rev'])

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.03, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=500,
             n_jobs=None, num_parallel_tree=None, ...)

In [22]:
test['rev_norm'] = test['rev_norm'].fillna(1)

residual_pred = model.predict(test[features_ml])

test['Revenue_final'] = test['Revenue_pred'] + residual_pred

In [23]:
train_ml['baseline_cogs'] = base_cogs * growth_cogs**(train_ml['year']-2022) * train_ml['cogs_norm']
train_ml['residual_cogs'] = train_ml['COGS'] - train_ml['baseline_cogs']

model_cogs = XGBRegressor(n_estimators=500, learning_rate=0.03)
model_cogs.fit(train_ml[features_ml], train_ml['residual_cogs'])

residual_cogs_pred = model_cogs.predict(test[features_ml])

test['COGS_final'] = test['COGS_pred'] + residual_cogs_pred

In [24]:
submission = pd.DataFrame({
    'Date': test['Date'].dt.strftime('%Y-%m-%d'),
    'Revenue': test['Revenue_final'].round(2),
    'COGS': test['COGS_final'].round(2)
})

submission.to_csv('submission.csv', index=False)

print(submission.head())

Empty DataFrame
Columns: [Date, Revenue, COGS]
Index: []


In [ ]:
base_rev  = annual.loc[2022, 'Revenue'] / 365
base_cogs = annual.loc[2022, 'COGS'] / 365

test = test.merge(seasonal, on=['month','day'], how='left')

test['rev_norm'] = test['rev_norm'].fillna(1)
test['cogs_norm'] = test['cogs_norm'].fillna(1)

test['years_ahead'] = test['year'] - 2022

test['Revenue_pred'] = base_rev * growth_rev**test['years_ahead'] * test['rev_norm']
test['COGS_pred'] = base_cogs * growth_cogs**test['years_ahead'] * test['cogs_norm']

## 2 — Load & Inspect Data

In [ ]:
train = pd.read_csv(TRAIN_FILE, parse_dates=['Date'])
test  = pd.read_csv(TEST_FILE,  parse_dates=['Date'])

print('Train shape:', train.shape)
print('Train date range:', train['Date'].min().date(), '→', train['Date'].max().date())
print()
print('Test shape:', test.shape)
print('Test date range:', test['Date'].min().date(), '→', test['Date'].max().date())
print()
train.tail()

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
axes[0].plot(train['Date'], train['Revenue'], lw=0.7)
axes[0].set_title('Historical Daily Revenue'); axes[0].set_ylabel('Revenue')
axes[1].plot(train['Date'], train['COGS'], lw=0.7, color='orange')
axes[1].set_title('Historical Daily COGS'); axes[1].set_ylabel('COGS')
plt.tight_layout()
plt.show()

## 3 — Feature Engineering

In [ ]:
train['year']       = train['Date'].dt.year
train['day_of_year'] = train['Date'].dt.dayofyear
train['month']      = train['Date'].dt.month
train['day']        = train['Date'].dt.day

# Annual totals — used to estimate YoY growth
annual = train.groupby('year')[['Revenue', 'COGS']].sum()
print('Annual totals (only complete years shown):')
print(annual)

In [ ]:
# --- YoY growth rate (geometric mean, 2013–2022) ---
# Use years with full data: 2013 to 2022
full_years = annual.loc[2013:2022]

yoy_rev  = full_years['Revenue'].pct_change().dropna()
yoy_cogs = full_years['COGS'].pct_change().dropna()

growth_rev  = (1 + yoy_rev).prod() ** (1 / len(yoy_rev))
growth_cogs = (1 + yoy_cogs).prod() ** (1 / len(yoy_cogs))

print(f'Geometric mean YoY Revenue growth : {growth_rev:.4f}  ({(growth_rev-1)*100:.2f}%/yr)')
print(f'Geometric mean YoY COGS    growth : {growth_cogs:.4f}  ({(growth_cogs-1)*100:.2f}%/yr)')

In [ ]:
# ===== TIME FEATURES =====
for df in [train, test]:
    df['dow'] = df['Date'].dt.dayofweek
    df['week'] = df['Date'].dt.isocalendar().week.astype(int)
    df['quarter'] = df['Date'].dt.quarter
    df['is_weekend'] = (df['dow'] >= 5).astype(int)

# ===== LAG FEATURES (from train only) =====
train = train.sort_values('Date')

for lag in [1, 7, 14, 30]:
    train[f'lag_rev_{lag}'] = train['Revenue'].shift(lag)
    train[f'lag_cogs_{lag}'] = train['COGS'].shift(lag)

# rolling (FIX leakage bằng shift)
train['roll7_rev'] = train['Revenue'].rolling(7).mean().shift(1)
train['roll30_rev'] = train['Revenue'].rolling(30).mean().shift(1)

# drop NA do lag
train_ml = train.dropna().copy()

## 4 — Build Seasonal Profile

Average Revenue / COGS by **(month, day)** across all available years. This captures seasonal patterns while smoothing out year-specific noise.

In [ ]:
# Normalise each year so seasonal profile is scale-free
annual_means = train.groupby('year')[['Revenue','COGS']].transform('mean')
train['rev_norm']  = train['Revenue'] / annual_means['Revenue']
train['cogs_norm'] = train['COGS']    / annual_means['COGS']

# Average normalised value for each (month, day)
seasonal = (
    train
    .groupby(['month', 'day'])[['rev_norm', 'cogs_norm']]
    .mean()
    .reset_index()
)

print('Seasonal profile rows:', len(seasonal))
seasonal.head(10)

## 5 — Predict Test Period

In [ ]:
# Base level: 2022 annual mean (most recent complete year)
base_rev  = annual.loc[2022, 'Revenue']  / 365
base_cogs = annual.loc[2022, 'COGS']     / 365

# How many years ahead of 2022 is each test date?
test = test.copy()
test['month'] = test['Date'].dt.month
test['day']   = test['Date'].dt.day
test['year']  = test['Date'].dt.year
test['years_ahead'] = test['year'] - 2022

# Merge seasonal profile
test = test.merge(seasonal, on=['month', 'day'], how='left')

# Fill any missing day (e.g. Feb-29 in non-leap years) with 1.0
test['rev_norm']  = test['rev_norm'].fillna(1.0)
test['cogs_norm'] = test['cogs_norm'].fillna(1.0)

# Predicted value = base_level × growth^years_ahead × seasonal_factor
test['Revenue_pred'] = (base_rev  * growth_rev**test['years_ahead']  * test['rev_norm'] ).round(2)
test['COGS_pred']    = (base_cogs * growth_cogs**test['years_ahead'] * test['cogs_norm']).round(2)

print('Predictions sample:')
test[['Date','Revenue_pred','COGS_pred']].head(10)

In [ ]:
# merge baseline vào train
train_ml = train_ml.merge(
    seasonal, on=['month','day'], how='left'
)

train_ml['rev_norm'] = train_ml['rev_norm'].fillna(1.0)
train_ml['baseline_rev'] = (
    base_rev * growth_rev**(train_ml['year'] - 2022) * train_ml['rev_norm']
)

train_ml['residual_rev'] = train_ml['Revenue'] - train_ml['baseline_rev']

In [ ]:
features_ml = [
    'dow','week','month','quarter','is_weekend',
    'lag_rev_1','lag_rev_7','lag_rev_14','lag_rev_30',
    'roll7_rev','roll30_rev'
]

In [ ]:
from xgboost import XGBRegressor

X_train_ml = train_ml[features_ml]
y_train_ml = train_ml['residual_rev']

model_res = XGBRegressor(
    n_estimators=500,
    learning_rate=0.03,
    max_depth=6
)

model_res.fit(X_train_ml, y_train_ml)

In [ ]:
# dùng lag cuối cùng từ train
last_values = train.tail(30).copy()

# fake lag cho test (baseline)
for lag in [1,7,14,30]:
    test[f'lag_rev_{lag}'] = last_values['Revenue'].iloc[-lag]

test['roll7_rev'] = last_values['Revenue'].tail(7).mean()
test['roll30_rev'] = last_values['Revenue'].tail(30).mean()

In [ ]:
X_test_ml = test[features_ml]
residual_pred = model_res.predict(X_test_ml)

In [ ]:
test['Revenue_final'] = test['Revenue_pred'] + residual_pred

In [ ]:
train_ml['baseline_cogs'] = (
    base_cogs * growth_cogs**(train_ml['year'] - 2022) * train_ml['cogs_norm']
)

train_ml['residual_cogs'] = train_ml['COGS'] - train_ml['baseline_cogs']

y_train_cogs = train_ml['residual_cogs']

model_cogs = XGBRegressor(n_estimators=500, learning_rate=0.03)
model_cogs.fit(X_train_ml, y_train_cogs)

residual_cogs_pred = model_cogs.predict(X_test_ml)

test['COGS_final'] = test['COGS_pred'] + residual_cogs_pred

In [ ]:
submission = pd.DataFrame({
    'Date': test['Date'].dt.strftime('%Y-%m-%d'),
    'Revenue': test['Revenue_final'].round(2),
    'COGS': test['COGS_final'].round(2)
})

submission.to_csv(OUT_FILE, index=False)

print("DONE ✅")
print(submission.head())

## 6 — Evaluate on Training Tail (2021–2022)

Quick sanity-check: apply the same method on the last 2 years of training data and measure MAPE.

In [ ]:
val = train[train['year'].isin([2021, 2022])].copy()
val = val.merge(seasonal, on=['month', 'day'], how='left')
val['rev_norm']  = val['rev_norm'].fillna(1.0)
val['cogs_norm'] = val['cogs_norm'].fillna(1.0)
val['years_ahead'] = val['year'] - 2022  # negative for historical
val['Revenue_pred'] = base_rev  * growth_rev**val['years_ahead']  * val['rev_norm']
val['COGS_pred']    = base_cogs * growth_cogs**val['years_ahead'] * val['cogs_norm']

def mape(actual, pred):
    return (np.abs(actual - pred) / actual).mean() * 100

print(f'MAPE Revenue (2021–2022): {mape(val["Revenue"], val["Revenue_pred"]):.2f}%')
print(f'MAPE COGS    (2021–2022): {mape(val["COGS"],    val["COGS_pred"]):.2f}%')

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(val['Date'], val['Revenue'],      lw=0.8, label='Actual')
ax.plot(val['Date'], val['Revenue_pred'], lw=0.8, linestyle='--', label='Predicted')
ax.set_title('Revenue — Actual vs Predicted (2021–2022 validation)')
ax.legend(); plt.tight_layout(); plt.show()

## 7 — Export Submission

In [ ]:
submission = test[['Date', 'Revenue_pred', 'COGS_pred']].rename(
    columns={'Revenue_pred': 'Revenue', 'COGS_pred': 'COGS'}
)
submission['Date'] = submission['Date'].dt.strftime('%Y-%m-%d')
submission.to_csv(OUT_FILE, index=False)

print(f'Saved {len(submission)} rows to {OUT_FILE}')
submission.head(10)